In [ ]:
# # %%capture
# !pip install --upgrade transformers bitsandbytes accelerate peft datasets wandb contractions spacy tqdm nltk
# !pip install --upgrade contractions matplotlib seaborn numpy

# !pip install --upgrade scikit-learn --no-cache-dir
# !python -m spacy download en_core_web_sm
# print("Done")

In [ ]:
import torch
import wandb
import pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset, DatasetDict
import numpy as np
import random
import re
from sklearn.metrics import precision_score, f1_score, recall_score, accuracy_score, balanced_accuracy_score
from sklearn.model_selection import train_test_split
from contractions import fix
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE, trustworthiness
from transformers import AutoModel
from torch.utils.data import DataLoader


# Login to Weights & Biases (W&B) - Required in Kaggle
wandb.login(key="REDACTED")  # Replace with your actual API key
wandb.init(project="Spring2025")

# Ensure GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} for training")

In [ ]:
import spacy

# Load spaCy model with GPU support
# spacy.require_gpu()  # Ensure spaCy runs on GPU

# Verify if GPU is used
print("spaCy is using GPU:", spacy.prefer_gpu())

# Load spaCy's English NLP model
nlp = spacy.load("en_core_web_sm")

def preprocess_text(text):
    """
    Preprocess legal text for analysis using spaCy.
    Steps:
    - Lowercasing
    - Removing HTML tags
    - Removing URLs
    - Expanding contractions
    - Removing special characters
    - Removing numbers
    - Removing stop words
    - Lemmatization using spaCy
    - Removing extra whitespace
    """
    # Lowercase the text
    text = text.lower()
    
    # Remove HTML tags and URLs
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    
    # Expand contractions (optional)
    try:
        text = fix(text)
    except ImportError:
        pass  # Skip if the 'contractions' package is not available
    
    # Remove special characters and numbers (excluding spaces)
    text = re.sub(r'[^a-z\s]', '', text)

    # Process text using spaCy
    doc = nlp(text)
    
    # Lemmatize and remove stopwords
    text = ' '.join([token.lemma_ for token in doc if not token.is_stop])

    # Remove extra whitespace
    text = ' '.join(text.split())
    
    return text



# Load dataset from CSV
df = pd.read_csv("archive/15_labels_data.csv")  # Adjust file path if needed
print(df.head())

In [ ]:
# Base sentence pool (Wikipedia-like)
base_sentences = [
    "The economy is recovering after a global slowdown.",
    "Scientists have discovered a new species in the Amazon.",
    "A policy to combat climate change was introduced.",
    "NASA released new images from the James Webb telescope.",
    "Research shows a link between diet and mental health.",
    "The EU is updating its data protection regulations.",
    "Artificial intelligence is transforming healthcare.",
    "Electric vehicle sales reached a record high.",
    "Astronomers detected signals from a distant galaxy.",
    "Educational reforms are being debated in Parliament."
]

# Generate 1000 rows
rows = []
for i in range(1000):
    sentence = random.choice(base_sentences) + f" Sentence {i}."
    label = 16  # Simulating labels 0–9
    rows.append([i, sentence, label])

# Create DataFrame with matching column names
df_simulated = pd.DataFrame(rows, columns=["Unnamed: 0", "text", "label"])

# Apply preprocessing
df_combined = pd.concat([df, df_simulated], ignore_index=True)
# df_combined = df_combined[:10]

df_combined.head()

processed_texts = [preprocess_text(text) for text in tqdm(df_combined['text'], desc="Preprocessing texts")]
df_combined['preprocessed_text'] = processed_texts

# Save df_combined to a CSV file
df_combined.to_csv('df_combined.csv', index=False)


In [ ]:
df = pd.read_csv('df_combined.csv')
# Step 1: Convert DataFrame to HF dataset
dataset = Dataset.from_pandas(df)

# Step 2: Train/Test Split
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset['train']
eval_dataset = dataset['test']

model_legalBERT = "nlpaueb/legal-bert-base-uncased"
model_DeepSeek = "deepseek-ai/deepseek-llm-7b-base"

# Step 3: Define tokenization functions for each model
def tokenize_legalbert(example):
    tokenizer = AutoTokenizer.from_pretrained(model_legalBERT)
    return tokenizer(example["preprocessed_text"], truncation=True, padding="max_length", max_length=512)

def tokenize_deepseek(example):
    tokenizer = AutoTokenizer.from_pretrained(model_DeepSeek)
    return tokenizer(example["preprocessed_text"], truncation=True, padding="max_length", max_length=512)

# Step 4: Apply tokenization
train_legalbert = train_dataset.map(tokenize_legalbert, batched=True)
eval_legalbert = eval_dataset.map(tokenize_legalbert, batched=True)

train_deepseek = train_dataset.map(tokenize_deepseek, batched=True)
eval_deepseek = eval_dataset.map(tokenize_deepseek, batched=True)

In [ ]:
# Count number of non-padding tokens
train_legalbert = train_legalbert.map(lambda x: {"token_count": sum(x["attention_mask"])})
train_deepseek = train_deepseek.map(lambda x: {"token_count": sum(x["attention_mask"])})

# Convert to pandas
df_legalbert = train_legalbert.to_pandas()
df_deepseek = train_deepseek.to_pandas()

# Plot histograms
plt.figure(figsize=(12, 6))
plt.hist(df_legalbert["token_count"], bins=30, alpha=0.6, label="LegalBERT")
plt.hist(df_deepseek["token_count"], bins=30, alpha=0.6, label="DeepSeek")
plt.title("Token Count Distribution per Sample")
plt.xlabel("Token Count")
plt.ylabel("Frequency")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Check average token length
print("LegalBERT avg:", df_legalbert["token_count"].mean())
print("DeepSeek avg:", df_deepseek["token_count"].mean())

plt.xlim(0, 200)  # or adjust based on your data

import seaborn as sns

sns.ecdfplot(df_legalbert["token_count"], label="LegalBERT")
sns.ecdfplot(df_deepseek["token_count"], label="DeepSeek")
plt.title("Cumulative Token Distribution")
plt.xlabel("Token Count")
plt.ylabel("CDF")
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
df_diff = df_legalbert["token_count"] - df_deepseek["token_count"]
plt.figure(figsize=(12, 6))
plt.hist(df_diff, bins=30, alpha=0.6)
plt.title("Token Count Difference Distribution")
plt.xlabel("Token Count Difference")
plt.ylabel("Frequency")
plt.grid(True)
plt.show()

In [9]:
from transformers import AutoModel
from torch.utils.data import DataLoader
from sklearn.manifold import TSNE, trustworthiness
import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm import tqdm

def extract_cls_tsne_projection(model_name, dataset, device="cpu", batch_size=16, label_field="label"):
    # Load model
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    # Prepare dataset and dataloader
    dataset.set_format(type="torch", columns=["input_ids", "attention_mask", label_field])
    loader = DataLoader(dataset, batch_size=batch_size)

    cls_embeddings = []
    labels = []

    for batch in tqdm(loader, desc=f"Extracting CLS embeddings for {model_name}"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            # cls_output = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            cls_output = outputs.last_hidden_state[:, 0, :].detach().cpu().tolist()
        cls_embeddings.extend(cls_output)
        labels.extend(batch[label_field].detach().cpu().tolist())

    cls_embeddings = np.array(cls_embeddings)
    labels = np.array(labels)

    # t-SNE
    tsne = TSNE(n_components=2, perplexity=50, learning_rate=300, n_iter=2000, random_state=42)
    tsne_result = tsne.fit_transform(cls_embeddings)

    # Trustworthiness
    trust = trustworthiness(cls_embeddings, tsne_result, n_neighbors=5)
    print(f"[{model_name}] Trustworthiness of t-SNE projection: {trust:.4f}")

    # Plot
    plt.figure(figsize=(10, 6))
    scatter = plt.scatter(tsne_result[:, 0], tsne_result[:, 1], c=labels, cmap='tab20', alpha=0.7)
    plt.legend(*scatter.legend_elements(), title="Labels", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.title(f"{model_name} – t-SNE Projection (CLS) with Label Coloring")
    plt.xlabel("Dimension 1")
    plt.ylabel("Dimension 2")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    return cls_embeddings, tsne_result, labels


In [ ]:
emb_legalbert, tsne_legalbert, labels_legalbert = extract_cls_tsne_projection(model_legalBERT, train_legalbert)

In [ ]:
emb_legalbert, tsne_legalbert, labels_legalbert = extract_cls_tsne_projection(model_DeepSeek, train_deepseek)